# 05. Stationary alignment closure

This notebook compares angular closures and identifies the stationary von--Mises alignment bias as the most stable interpretable closure for preserving the aligned-strain source.


# Ellipsoid stationary alignment closure, v14

This notebook is the next step after v13. The v13 result showed that the aspect-ratio dynamics is controlled by the alignment source

$$
2A\cos\alpha,
$$

where

$$
A=|S|,\qquad \alpha=2(\theta_S-\theta_g).
$$

The direct short-time regression of $\dot\alpha$ was useful, but noisy. The added VM experiment showed that fitting the stationary alignment bias is more stable. This notebook therefore promotes the stationary alignment closure to the main model.

The state variables are

$$
x=(v,\sigma,A,\omega,\alpha),\qquad r=e^v.
$$

The empirical train is the MEE / averaged-gradient train only:

$$
(M,g)=(M^{(\mathrm{avg;out})},g^{(\mathrm{out})}).
$$

We compare three closures:

1. **F baseline:** the v13 Fourier drift model for $\alpha$.
2. **VM:** stationary von-Mises alignment with $K(r,\sigma,A)$ inferred from $\mathbb E[\cos\alpha\mid r,\sigma,A]$.
3. **VMw:** VM plus a weak vorticity-driven rotation term.

The key validation quantities are

$$
\langle \sigma\mid r\rangle,
\qquad
\langle \cos\alpha\mid r\rangle,
\qquad
\langle 2A\cos\alpha\mid r\rangle.
$$

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import i0, i1

OUT_DIR = "figs_stationary_alignment_v14"
os.makedirs(OUT_DIR, exist_ok=True)

DATA_PATH = "empirical_train_mavg_out_v10b_enriched.npz"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "/mnt/data/empirical_train_mavg_out_v10b_enriched.npz"

rng = np.random.default_rng(1400)
np.set_printoptions(precision=4, suppress=True)

print("Loading", DATA_PATH)
z = np.load(DATA_PATH)
print("Keys:", z.files)

## 1. Load the empirical train and construct intrinsic variables

We use the intrinsic variables

$$
A=\sqrt{s_+^2+s_\times^2},
\qquad
\alpha=2(\theta_S-\theta_g)\mod 2\pi.
$$

The absolute angle $\theta_g$ should be irrelevant by isotropy. The relative angle $\alpha$ is the meaningful alignment variable.

In [ ]:
def wrap_pi(x):
    return (x + np.pi) % (2*np.pi) - np.pi

def bin_index_from_r(rval, edges):
    return np.searchsorted(edges, rval, side="right") - 1

times = z["times"]
dt = float(np.median(np.diff(times)))
r_edges = z["r_edges"]
r_centers = z["r_centers"]
n_bins = len(r_centers)

s_plus = z["s_plus"]
s_cross = z["s_cross"]
omega = z["omega"]
v = z["v"]
sigma = z["sigma"]
theta_g = z["theta_g"]
theta_s = z["theta_s"]
r = z["r"]

A = np.sqrt(s_plus**2 + s_cross**2)
alpha = wrap_pi(2*(theta_s - theta_g))
q = A*np.cos(alpha)

n_seed, n_time = sigma.shape
print("n_seed, n_time:", n_seed, n_time)
print("dt:", dt)
print("r bins:", r_centers)
print("sigma range:", float(np.nanmin(sigma)), float(np.nanmax(sigma)))

## 2. Short-time increments

We estimate empirical generator components from a short lag $\Delta$. The angular increment is wrapped:

$$
\Delta\alpha=\operatorname{atan2}(\sin(\alpha_{t+\Delta}-\alpha_t),\cos(\alpha_{t+\Delta}-\alpha_t)).
$$

The lag is deliberately longer than one numerical time step to reduce derivative noise, but still short relative to the evolution time.

In [ ]:
lag_steps = 5
lag_dt = lag_steps*dt
sl0 = np.s_[:, :-lag_steps]
sl1 = np.s_[:, lag_steps:]

v0, v1 = v[sl0], v[sl1]
s0, s1 = sigma[sl0], sigma[sl1]
A0, A1 = A[sl0], A[sl1]
w0, w1 = omega[sl0], omega[sl1]
a0, a1 = alpha[sl0], alpha[sl1]
r0 = r[sl0]

bv_emp = (v1-v0)/lag_dt
bA_emp = (A1-A0)/lag_dt
bw_emp = (w1-w0)/lag_dt
bs_emp = (s1-s0)/lag_dt
ba_emp = wrap_pi(a1-a0)/lag_dt

source0 = 2*A0*np.cos(a0)
Rsig_emp = bs_emp - source0

# Flattened arrays used for regression.
rf = r0.ravel()
vf = v0.ravel()
sf = s0.ravel()
Af = A0.ravel()
wf = w0.ravel()
af = a0.ravel()
qf = (A0*np.cos(a0)).ravel()

bvf = bv_emp.ravel()
bAf = bA_emp.ravel()
bwf = bw_emp.ravel()
bsf = bs_emp.ravel()
baf = ba_emp.ravel()
Rsf = Rsig_emp.ravel()
sourcef = source0.ravel()

binf = bin_index_from_r(rf, r_edges)
valid = np.isfinite(rf+sf+Af+wf+af+baf+Rsf) & (binf >= 0) & (binf < n_bins)
print("valid increment samples:", int(valid.sum()))

## 3. Utility routines

We use simple, stable estimators: radial bin means, ridge fits, and linear interpolation in $r$. This notebook is intended to test model structure, not optimize every parameter.

In [ ]:
def bin_mean(x, b, n_bins, min_count=10):
    out = np.full(n_bins, np.nan)
    cnt = np.zeros(n_bins, dtype=int)
    for k in range(n_bins):
        sel = (b == k) & np.isfinite(x)
        cnt[k] = int(sel.sum())
        if cnt[k] >= min_count:
            out[k] = np.nanmean(x[sel])
    return out, cnt

def bin_var(x, b, n_bins, min_count=10):
    out = np.full(n_bins, np.nan)
    for k in range(n_bins):
        sel = (b == k) & np.isfinite(x)
        if sel.sum() >= min_count:
            out[k] = np.nanvar(x[sel])
    return out

def fill_nan(arr):
    arr = np.asarray(arr, float).copy()
    idx = np.arange(len(arr))
    good = np.isfinite(arr)
    if good.sum() == 0:
        return np.zeros_like(arr)
    if good.sum() == 1:
        arr[~good] = arr[good][0]
        return arr
    arr[~good] = np.interp(idx[~good], idx[good], arr[good])
    return arr

def interp_by_r(rval, arr):
    arr = fill_nan(arr)
    return np.interp(np.asarray(rval), r_centers, arr, left=arr[0], right=arr[-1])

def ridge_fit(X, y, lam=1e-6):
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    ok = np.all(np.isfinite(X), axis=1) & np.isfinite(y)
    X = X[ok]
    y = y[ok]
    if len(y) < X.shape[1] + 5:
        return np.zeros(X.shape[1]), np.nan, len(y)
    XtX = X.T @ X
    Xty = X.T @ y
    reg = lam*np.eye(X.shape[1])
    reg[0,0] = 0.0
    beta = np.linalg.solve(XtX + reg, Xty)
    resid = y - X @ beta
    return beta, float(np.mean(resid**2)), len(y)

def A1_of_K(K):
    K = np.asarray(K, float)
    return i1(K) / np.maximum(i0(K), 1e-300)

def invert_A1(m, max_iter=30):
    m = np.asarray(m, float)
    m_clip = np.clip(m, 0.0, 0.985)
    K = np.empty_like(m_clip)
    mask1 = m_clip < 0.53
    mask2 = (m_clip >= 0.53) & (m_clip < 0.85)
    mask3 = m_clip >= 0.85
    K[mask1] = 2*m_clip[mask1] + m_clip[mask1]**3 + 5*m_clip[mask1]**5/6
    K[mask2] = -0.4 + 1.39*m_clip[mask2] + 0.43/(1 - m_clip[mask2])
    K[mask3] = 1/(m_clip[mask3]**3 - 4*m_clip[mask3]**2 + 3*m_clip[mask3] + 1e-12)
    for _ in range(max_iter):
        A1 = A1_of_K(K)
        der = 1 - A1/np.maximum(K, 1e-8) - A1**2
        K = K - (A1 - m_clip)/np.maximum(der, 1e-8)
        K = np.clip(K, 0.0, 100.0)
    return K

## 4. Empirical validation statistics

The target statistics are computed from the empirical train. These are not fitting targets for every parameter; they are used to validate the forward simulations.

In [ ]:
def empirical_by_bin():
    b = bin_index_from_r(r.ravel(), r_edges)
    sig = sigma.ravel()
    Aval = A.ravel()
    aval = alpha.ravel()
    qval = Aval*np.cos(aval)
    out = {}
    out["sigma"], out["count"] = bin_mean(sig, b, n_bins, min_count=10)
    out["A"], _ = bin_mean(Aval, b, n_bins, min_count=10)
    out["cos"], _ = bin_mean(np.cos(aval), b, n_bins, min_count=10)
    out["source"], _ = bin_mean(2*qval, b, n_bins, min_count=10)
    out["q"], _ = bin_mean(qval, b, n_bins, min_count=10)
    return out

emp = empirical_by_bin()
print("Empirical sigma:", emp["sigma"])
print("Empirical cos alpha:", emp["cos"])
print("Empirical source:", emp["source"])
print("Counts:", emp["count"])

## 5. Common closures for $v$, $A$, $\omega$, and $R_\sigma$

The alignment model is the focus, but it must be embedded in a complete forward simulation. We keep simple radial closures for the remaining variables.

For $A$ and $\omega$, we use OU-like radial closures. For the aspect-ratio residual we fit

$$
R_\sigma(r,\sigma)=c_\sigma(r)-\frac{1}{2}\kappa(r)\sinh(2\sigma).
$$

The shape equation is then

$$
d\sigma=\left[2A\cos\alpha+c_\sigma(r)-\frac{1}{2}\kappa(r)\sinh(2\sigma)\right]dt+\sqrt{2D_\sigma(r)}dW_\sigma.
$$

In [ ]:
b_v = np.full(n_bins, np.nan); D_v = np.full(n_bins, np.nan)
A_bar = np.full(n_bins, np.nan); lambda_A = np.full(n_bins, np.nan); D_A = np.full(n_bins, np.nan)
lambda_w = np.full(n_bins, np.nan); D_w = np.full(n_bins, np.nan)
c_sigma = np.full(n_bins, np.nan); kappa = np.full(n_bins, np.nan); D_sigma = np.full(n_bins, np.nan)
D_alpha = np.full(n_bins, np.nan)

for k in range(n_bins):
    sel = valid & (binf == k)
    if sel.sum() < 50:
        continue
    # v drift/diffusion
    b_v[k] = np.mean(bvf[sel])
    D_v[k] = 0.5*np.var((v1-v0).ravel()[sel]) / lag_dt

    # A OU closure
    A00 = Af[sel]
    A11 = A1.ravel()[sel]
    A_bar[k] = np.mean(A00)
    x0c = A00 - A_bar[k]
    x1c = A11 - A_bar[k]
    var0 = np.var(x0c)
    if var0 > 1e-12:
        rhoA = np.mean(x0c*x1c)/var0
        rhoA = np.clip(rhoA, 1e-4, 0.999)
        lambda_A[k] = -np.log(rhoA)/lag_dt
        D_A[k] = lambda_A[k]*var0
    else:
        lambda_A[k] = 1.0; D_A[k] = 1e-8

    # omega OU closure, zero mean
    w00 = wf[sel]
    w11 = w1.ravel()[sel]
    varw = np.var(w00)
    if varw > 1e-12:
        rhow = np.mean(w00*w11)/varw
        rhow = np.clip(rhow, 1e-4, 0.999)
        lambda_w[k] = -np.log(rhow)/lag_dt
        D_w[k] = lambda_w[k]*varw
    else:
        lambda_w[k] = 1.0; D_w[k] = 1e-8

    # R_sigma residual
    X = np.column_stack([np.ones(sel.sum()), np.sinh(2*np.clip(sf[sel], 0, 5))])
    beta, mse, n = ridge_fit(X, Rsf[sel], lam=1e-4)
    c_sigma[k] = beta[0]
    kappa[k] = max(0.0, -2*beta[1])
    pred = X @ beta
    D_sigma[k] = 0.5*np.var((Rsf[sel] - pred)*lag_dt) / lag_dt

    # alpha diffusion from increments; drift will be model-dependent.
    D_alpha[k] = 0.5*np.var(wrap_pi(a1-a0).ravel()[sel]) / lag_dt

print("b_v:", b_v)
print("A_bar:", A_bar)
print("lambda_A:", lambda_A)
print("lambda_w:", lambda_w)
print("c_sigma:", c_sigma)
print("kappa:", kappa)
print("D_alpha:", D_alpha)

## 6. Baseline F: Fourier drift closure for $\alpha$

This is the v13 baseline. It directly regresses the short-time angular drift using periodic features. It is useful as a comparison, but the v13 result suggested that direct drift regression is noisy.

In [ ]:
def alpha_features(sig, Aval, wval, aval):
    return np.column_stack([
        np.ones_like(aval),
        wval,
        np.sin(aval), np.cos(aval),
        np.sin(2*aval), np.cos(2*aval),
        sig*np.sin(aval), sig*np.cos(aval),
        Aval*np.sin(aval), Aval*np.cos(aval),
    ])

alpha_beta = np.zeros((n_bins, 10))
alpha_mse = np.full(n_bins, np.nan)

for k in range(n_bins):
    sel = valid & (binf == k)
    if sel.sum() < 100:
        continue
    X = alpha_features(sf[sel], Af[sel], wf[sel], af[sel])
    beta, mse, n = ridge_fit(X, baf[sel], lam=5e-2)
    alpha_beta[k] = beta
    alpha_mse[k] = mse

print("alpha drift MSE by bin:", alpha_mse)
print("alpha coefficients first three bins:")
print(alpha_beta[:3])

## 7. Main v14 closure: stationary von-Mises alignment

The VM closure fits the conditional mean

$$
m_\alpha(r,\sigma,A)=\mathbb E[\cos\alpha\mid r,\sigma,A].
$$

We use a simple per-bin regression

$$
m_\alpha=m_0(r)+m_1(r)\sigma+m_2(r)A+m_3(r)\sigma A.
$$

This mean is converted to a von-Mises concentration $K$ through

$$
m_\alpha=I_1(K)/I_0(K).
$$

The pure VM angular SDE is

$$
d\alpha=-D_\alpha(r)K(r,\sigma,A)\sin\alpha\,dt+\sqrt{2D_\alpha(r)}dW_\alpha.
$$

In [ ]:
vm_beta = np.zeros((n_bins, 4))
vm_mse = np.full(n_bins, np.nan)
vm_K_mean = np.full(n_bins, np.nan)

for k in range(n_bins):
    sel = valid & (binf == k)
    if sel.sum() < 100:
        continue
    X = np.column_stack([np.ones(sel.sum()), sf[sel], Af[sel], sf[sel]*Af[sel]])
    y = np.cos(af[sel])
    beta, mse, n = ridge_fit(X, y, lam=1e-3)
    vm_beta[k] = beta
    vm_mse[k] = mse
    mhat = np.clip(X @ beta, -0.95, 0.95)
    vm_K_mean[k] = np.mean(invert_A1(np.maximum(mhat, 0)))

print("VM beta by r-bin [1, sigma, A, sigma*A]:")
print(vm_beta)
print("VM fit MSE:", vm_mse)
print("Mean inferred K by bin:", vm_K_mean)

def mcos_vm(rval, sig, Aval):
    b0 = interp_by_r(rval, vm_beta[:,0])
    b1 = interp_by_r(rval, vm_beta[:,1])
    b2 = interp_by_r(rval, vm_beta[:,2])
    b3 = interp_by_r(rval, vm_beta[:,3])
    return np.clip(b0 + b1*sig + b2*Aval + b3*sig*Aval, -0.95, 0.95)

def K_vm(rval, sig, Aval):
    return invert_A1(np.maximum(mcos_vm(rval, sig, Aval), 0.0))

## 8. VMw: VM plus vorticity-driven rotation

The pure VM closure enforces the stationary alignment bias but ignores circulating angular drift. We test a minimal correction:

$$
d\alpha=\left[c_0(r)+c_\omega(r)\omega-D_\alpha(r)K(r,\sigma,A)\sin\alpha\right]dt+\sqrt{2D_\alpha(r)}dW_\alpha.
$$

The coefficients $c_0$ and $c_\omega$ are fitted to the residual angular drift after subtracting the VM alignment force. This is an experiment, not yet a final model.

In [ ]:
rot_c0 = np.full(n_bins, np.nan)
rot_cw = np.full(n_bins, np.nan)
rot_mse = np.full(n_bins, np.nan)

for k in range(n_bins):
    sel = valid & (binf == k)
    if sel.sum() < 100:
        continue
    Daa = max(D_alpha[k], 1e-8)
    Khat = K_vm(rf[sel], sf[sel], Af[sel])
    residual = baf[sel] + Daa*Khat*np.sin(af[sel])
    X = np.column_stack([np.ones(sel.sum()), wf[sel]])
    beta, mse, n = ridge_fit(X, residual, lam=1e-1)
    rot_c0[k], rot_cw[k] = beta
    rot_mse[k] = mse

# Clip only for simulation stability, not for interpretation.
rot_c0_sim = np.clip(fill_nan(rot_c0), -5.0, 5.0)
rot_cw_sim = np.clip(fill_nan(rot_cw), -8.0, 8.0)

print("VMw residual rotation c0:", rot_c0)
print("VMw residual rotation c_omega:", rot_cw)
print("Clipped c_omega used in simulation:", rot_cw_sim)
print("Rotation residual MSE:", rot_mse)

## 9. Forward simulation

We now simulate three closures with identical radial closures:

- **F:** direct Fourier angular drift.
- **VM:** stationary von-Mises alignment.
- **VMw:** VM plus vorticity-driven rotation.

The models are evaluated only by forward statistics, not by one-step regression error.

In [ ]:
def alpha_drift_F(rval, sig, Aval, wval, aval):
    features = [
        np.ones_like(aval),
        wval,
        np.sin(aval), np.cos(aval),
        np.sin(2*aval), np.cos(2*aval),
        sig*np.sin(aval), sig*np.cos(aval),
        Aval*np.sin(aval), Aval*np.cos(aval),
    ]
    out = np.zeros_like(aval, dtype=float)
    for j, feat in enumerate(features):
        out += interp_by_r(rval, alpha_beta[:,j]) * feat
    return out

def simulate_model(kind="VM", n_sim=300, seed=0, T_steps=None):
    local_rng = np.random.default_rng(seed)
    if T_steps is None:
        T_steps = n_time

    # Use empirical t=0 initialization for v, sigma, A, omega.
    idx0 = local_rng.integers(0, n_seed, size=n_sim)
    vv = v[idx0, 0].copy()
    ss = sigma[idx0, 0].copy()
    AA = A[idx0, 0].copy()
    ww = omega[idx0, 0].copy()

    # For alpha, use a small-time empirical distribution rather than the ill-defined exactly isotropic t=0 angle.
    t_alpha_max = min(50, n_time)
    idx_seed = local_rng.integers(0, n_seed, size=n_sim)
    idx_time = local_rng.integers(0, t_alpha_max, size=n_sim)
    aa = alpha[idx_seed, idx_time].copy()

    out_v = np.zeros((n_sim, T_steps))
    out_sig = np.zeros_like(out_v)
    out_A = np.zeros_like(out_v)
    out_w = np.zeros_like(out_v)
    out_alpha = np.zeros_like(out_v)
    out_q = np.zeros_like(out_v)

    for t in range(T_steps):
        rr = np.exp(vv)
        qq = AA*np.cos(aa)
        out_v[:, t] = vv
        out_sig[:, t] = ss
        out_A[:, t] = AA
        out_w[:, t] = ww
        out_alpha[:, t] = aa
        out_q[:, t] = qq

        if t == T_steps - 1:
            break

        bv = interp_by_r(rr, b_v)
        Dv = np.maximum(interp_by_r(rr, D_v), 1e-10)

        Abar = interp_by_r(rr, A_bar)
        lamA = np.maximum(interp_by_r(rr, lambda_A), 0.0)
        DA = np.maximum(interp_by_r(rr, D_A), 1e-10)

        lamw = np.maximum(interp_by_r(rr, lambda_w), 0.0)
        Dw = np.maximum(interp_by_r(rr, D_w), 1e-10)

        cs = interp_by_r(rr, c_sigma)
        kap = np.maximum(interp_by_r(rr, kappa), 0.0)
        Ds = np.maximum(interp_by_r(rr, D_sigma), 1e-10)
        Daa = np.maximum(interp_by_r(rr, D_alpha), 1e-10)

        # v, A, omega updates.
        vv = vv + bv*dt + np.sqrt(2*Dv*dt)*local_rng.normal(size=n_sim)
        AA = AA + (-lamA*(AA - Abar))*dt + np.sqrt(2*DA*dt)*local_rng.normal(size=n_sim)
        AA = np.maximum(AA, 1e-6)
        ww = ww + (-lamw*ww)*dt + np.sqrt(2*Dw*dt)*local_rng.normal(size=n_sim)

        # alpha update.
        if kind == "F":
            drift_a = alpha_drift_F(rr, ss, AA, ww, aa)
        elif kind == "VM":
            drift_a = -Daa*K_vm(rr, ss, AA)*np.sin(aa)
        elif kind == "VMw":
            drift_a = (interp_by_r(rr, rot_c0_sim) + interp_by_r(rr, rot_cw_sim)*ww
                       - Daa*K_vm(rr, ss, AA)*np.sin(aa))
        else:
            raise ValueError(kind)

        aa = aa + drift_a*dt + np.sqrt(2*Daa*dt)*local_rng.normal(size=n_sim)
        aa = wrap_pi(aa)

        # sigma update.
        qq = AA*np.cos(aa)
        drift_sig = 2*qq + cs - 0.5*kap*np.sinh(2*np.clip(ss, 0, 5))
        ss = ss + drift_sig*dt + np.sqrt(2*Ds*dt)*local_rng.normal(size=n_sim)
        ss = np.clip(ss, 0.0, 5.0)

    return {"v": out_v, "r": np.exp(out_v), "sigma": out_sig, "A": out_A,
            "omega": out_w, "alpha": out_alpha, "q": out_q}

def diagnostics_from_sim(sim):
    rr = sim["r"].ravel()
    b = bin_index_from_r(rr, r_edges)
    out = {}
    out["sigma"], out["count"] = bin_mean(sim["sigma"].ravel(), b, n_bins, min_count=10)
    out["A"], _ = bin_mean(sim["A"].ravel(), b, n_bins, min_count=10)
    out["cos"], _ = bin_mean(np.cos(sim["alpha"].ravel()), b, n_bins, min_count=10)
    out["source"], _ = bin_mean(2*sim["A"].ravel()*np.cos(sim["alpha"].ravel()), b, n_bins, min_count=10)
    out["q"], _ = bin_mean(sim["A"].ravel()*np.cos(sim["alpha"].ravel()), b, n_bins, min_count=10)
    return out

sim_F = simulate_model("F", n_sim=300, seed=1411)
sim_VM = simulate_model("VM", n_sim=300, seed=1412)
sim_VMw = simulate_model("VMw", n_sim=300, seed=1413)

diag_F = diagnostics_from_sim(sim_F)
diag_VM = diagnostics_from_sim(sim_VM)
diag_VMw = diagnostics_from_sim(sim_VMw)

print("Empirical sigma:", emp["sigma"])
print("F sigma       :", diag_F["sigma"])
print("VM sigma      :", diag_VM["sigma"])
print("VMw sigma     :", diag_VMw["sigma"])
print("\nEmpirical source:", emp["source"])
print("F source       :", diag_F["source"])
print("VM source      :", diag_VM["source"])
print("VMw source     :", diag_VMw["source"])
print("\nEmpirical cos:", emp["cos"])
print("F cos       :", diag_F["cos"])
print("VM cos      :", diag_VM["cos"])
print("VMw cos     :", diag_VMw["cos"])

## 10. Validation plots

The most important diagnostic is the source ratio

$$
\frac{\langle 2A\cos\alpha\mid r\rangle_{\rm model}}{\langle 2A\cos\alpha\mid r\rangle_{\rm emp}}.
$$

If this ratio is too small, the model cannot sustain the observed aspect ratio.

In [ ]:
def plot_compare(key, ylabel, fname):
    plt.figure(figsize=(6.8, 4.5))
    plt.plot(r_centers, emp[key], "o-", label="empirical")
    plt.plot(r_centers, diag_F[key], "s--", label="F: Fourier drift")
    plt.plot(r_centers, diag_VM[key], "^-.", label="VM: stationary")
    plt.plot(r_centers, diag_VMw[key], "d-.", label="VMw: VM + omega rotation")
    plt.xscale("log")
    plt.xlabel("r")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = os.path.join(OUT_DIR, fname)
    plt.savefig(path, dpi=180)
    plt.show()
    print("saved", path)

plot_compare("sigma", r"$\langle\sigma\mid r\rangle$", "sigma_mean_comparison.png")
plot_compare("source", r"$\langle 2A\cos\alpha\mid r\rangle$", "source_comparison.png")
plot_compare("cos", r"$\langle\cos\alpha\mid r\rangle$", "cos_alpha_comparison.png")
plot_compare("A", r"$\langle A\mid r\rangle$", "A_mean_comparison.png")

eps = 1e-12
ratio_F = diag_F["source"] / np.maximum(emp["source"], eps)
ratio_VM = diag_VM["source"] / np.maximum(emp["source"], eps)
ratio_VMw = diag_VMw["source"] / np.maximum(emp["source"], eps)

print("Source ratios model/empirical:")
print("F  :", ratio_F)
print("VM :", ratio_VM)
print("VMw:", ratio_VMw)

plt.figure(figsize=(6.8, 4.2))
plt.axhline(1.0, color="k", lw=1, alpha=0.5)
plt.plot(r_centers, ratio_F, "s--", label="F")
plt.plot(r_centers, ratio_VM, "^-.", label="VM")
plt.plot(r_centers, ratio_VMw, "d-.", label="VMw")
plt.xscale("log")
plt.xlabel("r")
plt.ylabel(r"source ratio")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = os.path.join(OUT_DIR, "source_ratio.png")
plt.savefig(path, dpi=180)
plt.show()
print("saved", path)

## 11. Distribution checks

Mean agreement is not sufficient. We also compare distributions in representative bins. The VM closure should improve the bias of $\cos\alpha$ and the distribution of $\sigma$ in the middle inertial range.

In [ ]:
def hist_compare_quantity(quantity_emp, sim_quantities, labels, bin_ids, xlabel, fname_prefix, bins=40, value_range=None):
    b_emp = bin_index_from_r(r.ravel(), r_edges)
    q_emp = quantity_emp.ravel()
    for k in bin_ids:
        plt.figure(figsize=(6.8, 4.4))
        sel = (b_emp == k) & np.isfinite(q_emp)
        if sel.sum() > 20:
            plt.hist(q_emp[sel], bins=bins, range=value_range, density=True, alpha=0.35, label="empirical")
        for sim, lab in zip(sim_quantities, labels):
            b_sim = bin_index_from_r(sim["r"].ravel(), r_edges)
            if xlabel == r"$\sigma$":
                q_sim = sim["sigma"].ravel()
            elif xlabel == r"$\cos\alpha$":
                q_sim = np.cos(sim["alpha"].ravel())
            else:
                raise ValueError(xlabel)
            sel_s = (b_sim == k) & np.isfinite(q_sim)
            if sel_s.sum() > 20:
                plt.hist(q_sim[sel_s], bins=bins, range=value_range, density=True, histtype="step", lw=2, label=lab)
        plt.xlabel(xlabel)
        plt.ylabel("density")
        plt.title(f"r-bin {k}, r≈{r_centers[k]:.3g}")
        plt.grid(True, alpha=0.25)
        plt.legend()
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{fname_prefix}_bin{k}.png")
        plt.savefig(path, dpi=180)
        plt.show()
        print("saved", path)

representative_bins = [1, 3, 5]
hist_compare_quantity(sigma, [sim_F, sim_VM, sim_VMw], ["F", "VM", "VMw"], representative_bins,
                      r"$\sigma$", "sigma_hist", bins=35, value_range=(0, 2.5))
hist_compare_quantity(np.cos(alpha), [sim_F, sim_VM, sim_VMw], ["F", "VM", "VMw"], representative_bins,
                      r"$\cos\alpha$", "cosalpha_hist", bins=35, value_range=(-1, 1))

## 12. Reading the outcome

The outcome should be read as follows.

If VM improves $\langle\cos\alpha\mid r\rangle$ and the source ratio, then the stationary alignment distribution is more reliable than short-time angular drift regression.

If VMw improves further, then a circulating angular component associated with $\omega$ is needed. If VMw is unstable or worse, then the rotation term should be regularized or dropped.

The next model should keep the simplest closure that reproduces the alignment source and the $\sigma$ plateau.

In [ ]:
# Save results for interpretation and later reuse.
np.savez_compressed(
    "stationary_alignment_closure_v14_results.npz",
    r_edges=r_edges,
    r_centers=r_centers,
    b_v=b_v,
    D_v=D_v,
    A_bar=A_bar,
    lambda_A=lambda_A,
    D_A=D_A,
    lambda_w=lambda_w,
    D_w=D_w,
    c_sigma=c_sigma,
    kappa=kappa,
    D_sigma=D_sigma,
    D_alpha=D_alpha,
    alpha_beta=alpha_beta,
    vm_beta=vm_beta,
    vm_mse=vm_mse,
    vm_K_mean=vm_K_mean,
    rot_c0=rot_c0,
    rot_cw=rot_cw,
    emp_sigma=emp["sigma"],
    emp_A=emp["A"],
    emp_cos=emp["cos"],
    emp_source=emp["source"],
    F_sigma=diag_F["sigma"],
    F_A=diag_F["A"],
    F_cos=diag_F["cos"],
    F_source=diag_F["source"],
    VM_sigma=diag_VM["sigma"],
    VM_A=diag_VM["A"],
    VM_cos=diag_VM["cos"],
    VM_source=diag_VM["source"],
    VMw_sigma=diag_VMw["sigma"],
    VMw_A=diag_VMw["A"],
    VMw_cos=diag_VMw["cos"],
    VMw_source=diag_VMw["source"],
    ratio_F=ratio_F,
    ratio_VM=ratio_VM,
    ratio_VMw=ratio_VMw,
)
print("saved stationary_alignment_closure_v14_results.npz")
print("Notebook complete.")